# 🎓 Machine Learning Project: Resume Text Classification

### Part 1: Introduction
**Student Details:**
* First Name: [Your First Name]
* Last Name Initial: [Your Last Name Initial]
* Last 4 Digits of ID: [1234]

**AI Chatbot Prompts Used:**
* *Prompt 1:* "How to apply POS-aware lemmatization using NLTK in pandas..."
* *Prompt 2:* "How to extract feature importances from a multiclass Logistic Regression..."
* *Goal of usage:* To assist with syntax optimization and generating explainability visualizations.

**Problem & Dataset Explanation:**
This project focuses on **Text Classification** (Supervised Learning). The dataset used is the Resume Classification dataset from Kaggle, which contains thousands of raw resume text strings alongside their corresponding job category (e.g., Finance, IT, Healthcare). The goal is to build an NLP pipeline that cleans the raw text, extracts meaningful numerical features (using TF-IDF), and trains a machine learning model to accurately predict the job category of unseen resumes based purely on their textual content.


In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

sns.set_theme(style="whitegrid")


### Loading the Dataset and Splitting (Train/Test)
To absolutely prevent any data leakage, we will load the dataset and immediately split it into a `train_set` and `test_set` before applying any feature engineering or vectorization.


In [11]:
# Load dataset
df = pd.read_csv('Resume.csv')

# Basic cleaning of empty rows
if 'Resume_html' in df.columns:
    df = df.drop(columns=['Resume_html'])
df = df.dropna(subset=['Resume_str', 'Category'])
df = df.drop_duplicates(subset=['Resume_str']).reset_index(drop=True)
df['Category'] = df['Category'].str.strip().str.upper()

# Train-Test Split (80/20)
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Category'])

print(f"Train set shape: {df_train.shape}")
print(f"Test set shape: {df_test.shape}")

print("\nFirst 5 rows of Train Set:")
display(df_train.head(5))

print("\nFirst 5 rows of Test Set:")
display(df_test.head(5))


Train set shape: (1985, 3)
Test set shape: (497, 3)

First 5 rows of Train Set:


,ID,Resume_str,Category
2075,19503224,ASSOCIATE VICE PRESIDENT FOR COLLEGE ...,PUBLIC-RELATIONS
1856,63137898,ACCOUNTANT Summary Flexible ...,ACCOUNTANT
1482,27789372,FINANCE DIRECTOR Summary ...,FINANCE
836,25838512,OPERATIONS MANAGER Skills ...,FITNESS
182,34317538,LEAD INTERACTION DESIGNER Summa...,DESIGNER



First 5 rows of Test Set:


,ID,Resume_str,Category
1300,30504149,CHIEF DIGITAL OFFICER Summary ...,DIGITAL-MEDIA
27,29297973,HR REPRESENTATIVE Summary Ex...,HR
402,11616482,GUEST TEACHER Professional Over...,TEACHER
1050,34303500,SALES DIRECTOR Summary \n\n...,SALES
1140,22485475,CONSULTANT Professional Summary...,CONSULTANT


### Part 6D (Bonus): Special Adjustments for Imbalanced Data
The original dataset has 24 highly imbalanced and highly specific categories (e.g., 'Aviation' and 'Agriculture' have very few samples compared to 'Information-Technology'). 
To treat this imbalance robustly, we apply **Category Grouping** (Dimensionality Reduction of the Target Variable). We map the 24 classes into 9 broader, semantically balanced macro-categories.


In [12]:
category_mapping = {
    'INFORMATION-TECHNOLOGY': 'Technology',
    'BUSINESS-DEVELOPMENT': 'Business & Operations',
    'SALES': 'Business & Operations',
    'BPO': 'Business & Operations',
    'CONSULTANT': 'Business & Operations',
    'FINANCE': 'Finance & Accounting',
    'ACCOUNTANT': 'Finance & Accounting',
    'BANKING': 'Finance & Accounting',
    'ENGINEERING': 'Engineering & Infrastructure',
    'CONSTRUCTION': 'Engineering & Infrastructure',
    'AUTOMOBILE': 'Engineering & Infrastructure',
    'AVIATION': 'Engineering & Infrastructure',
    'DESIGNER': 'Creative & Media',
    'DIGITAL-MEDIA': 'Creative & Media',
    'PUBLIC-RELATIONS': 'Creative & Media',
    'ARTS': 'Creative & Media',
    'APPAREL': 'Creative & Media',
    'HEALTHCARE': 'Healthcare & Wellness',
    'FITNESS': 'Healthcare & Wellness',
    'TEACHER': 'Education',
    'ADVOCATE': 'Legal & HR',
    'HR': 'Legal & HR',
    'CHEF': 'Food & Agriculture',
    'AGRICULTURE': 'Food & Agriculture'
}

# Apply mapping to train and test
df_train['Macro_Category'] = df_train['Category'].map(category_mapping)
df_test['Macro_Category'] = df_test['Category'].map(category_mapping)

df_train = df_train.dropna(subset=['Macro_Category']).reset_index(drop=True)
df_test = df_test.dropna(subset=['Macro_Category']).reset_index(drop=True)

y_train = df_train['Macro_Category']
y_test = df_test['Macro_Category']


### Part 2: Feature Engineering
We will apply text normalization, POS-Aware Lemmatization, and TF-IDF vectorization.
We will explicitly show 3 examples from the train set going through the pipeline.


In [13]:
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag

# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('averaged_perceptron_tagger')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'): return wordnet.ADJ
    elif treebank_tag.startswith('V'): return wordnet.VERB
    elif treebank_tag.startswith('N'): return wordnet.NOUN
    elif treebank_tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def clean_text(text):
    text = str(text).lower()
    text = re.sub('http\S+\s*', ' ', text)
    text = re.sub('RT|cc', ' ', text)
    text = re.sub('#\S+', '', text)
    text = re.sub('@\S+', '  ', text)
    text = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', text)
    text = re.sub(r'[^\x00-\x7f]', r' ', text) 
    text = re.sub('\s+', ' ', text)
    
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]
    
    pos_tokens = pos_tag(tokens)
    cleaned = [lemmatizer.lemmatize(w, get_wordnet_pos(tag)) for w, tag in pos_tokens]
    return ' '.join(cleaned)

print("Applying Feature Engineering (Cleaning)...")
df_train['Cleaned'] = df_train['Resume_str'].apply(clean_text)
df_test['Cleaned'] = df_test['Resume_str'].apply(clean_text)

print("\nShowing 3 examples of Feature Engineering (Original vs Cleaned):")
for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print(f"ORIGINAL (first 100 chars): {df_train['Resume_str'].iloc[i][:100]}...")
    print(f"CLEANED  (first 100 chars): {df_train['Cleaned'].iloc[i][:100]}...")


Applying Feature Engineering (Cleaning)...

Showing 3 examples of Feature Engineering (Original vs Cleaned):

--- Example 1 ---
ORIGINAL (first 100 chars):          ASSOCIATE VICE PRESIDENT FOR COLLEGE ADVANCEMENT & PUBLIC RELATIONS         Executive Profi...
CLEANED  (first 100 chars): associate vice president college advancement public relation executive profile work high education a...

--- Example 2 ---
ORIGINAL (first 100 chars):          ACCOUNTANT       Summary    Flexible bookkeeper/ accountant who adapts seamlessly to consta...
CLEANED  (first 100 chars): ountant summary flexible bookkeeper ountant adapt seamlessly constantly evolve ounting process techn...

--- Example 3 ---
ORIGINAL (first 100 chars):          FINANCE DIRECTOR         Summary     Finance Director with experience in strategic planning...
CLEANED  (first 100 chars): finance director summary finance director experience strategic planning budget ounting highlight dat...


In [14]:
# TF-IDF Vectorization
# We fit ONLY on the train set to absolutely prevent data leakage
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train = tfidf.fit_transform(df_train['Cleaned'])
X_test = tfidf.transform(df_test['Cleaned'])

print(f"\nTF-IDF Train Matrix Shape: {X_train.shape}")
print(f"TF-IDF Test Matrix Shape: {X_test.shape}")



TF-IDF Train Matrix Shape: (1985, 5000)
TF-IDF Test Matrix Shape: (497, 5000)


### Part 3: Implementation of Learning Algorithms
We implement **Logistic Regression**, which is a linear model that assigns weights to each TF-IDF word feature. It is exceptionally well-suited for high-dimensional sparse matrices like TF-IDF, as it can efficiently learn linear boundaries between complex text classes without severely overfitting.

### Part 6A, 6C, 6G (Bonuses): Hyperparameter GridSearch, K-Fold CV, and Results Presentation
We use `GridSearchCV` with 5-Fold Stratified Cross-Validation (`cv=5`). We will test different hyperparameters for Logistic Regression, collect all permutations into a DataFrame, and display the best combination.


In [15]:
# Define hyperparameter grid for Logistic Regression
param_grid = {
    'C': [0.1, 1, 10],
    'solver': ['lbfgs', 'liblinear']
}

# K-Fold CV GridSearch
grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000, multi_class='auto'),
    param_grid,
    cv=5, 
    scoring='f1_macro', # Using macro f1 as requested for multiclass
    n_jobs=-1,
    return_train_score=False
)

print("Running 5-Fold Cross Validation GridSearch...")
grid_search.fit(X_train, y_train)

# Part 6G: Show all permutations in a DataFrame
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df[['param_C', 'param_solver', 'mean_test_score', 'std_test_score', 'rank_test_score']]
results_df = results_df.rename(columns={'mean_test_score': 'Mean_Macro_F1'})
results_df = results_df.sort_values(by='rank_test_score')

print("\nAll Hyperparameter Permutations and their K-Fold Macro F1 Scores:")
display(results_df)

print(f"\nBest Permutation: {grid_search.best_params_}")
print(f"Best Macro F1 Score (Validation): {grid_search.best_score_:.4f}")


Running 5-Fold Cross Validation GridSearch...

All Hyperparameter Permutations and their K-Fold Macro F1 Scores:


,param_C,param_solver,Mean_Macro_F1,std_test_score,rank_test_score
5,10.0,liblinear,0.719364,0.025931,1
4,10.0,lbfgs,0.717205,0.024026,2
3,1.0,liblinear,0.685267,0.041573,3
2,1.0,lbfgs,0.685208,0.037608,4
0,0.1,lbfgs,0.425046,0.029221,5
1,0.1,liblinear,0.421354,0.028320,6



Best Permutation: {'C': 10, 'solver': 'liblinear'}
Best Macro F1 Score (Validation): 0.7194


### Part 4: Training with the Best Combination
We now extract the best model from the GridSearch (which is already retrained on the entire `trainset` by default via `refit=True`) to be ready for testing.


In [16]:
best_model = grid_search.best_estimator_
print(f"Model selected for final testing: {best_model}")


Model selected for final testing: LogisticRegression(C=10, max_iter=1000, multi_class='auto', solver='liblinear')


### Part 6F (Bonus): Explainability (Feature Importance)
To understand what our Logistic Regression model actually learned, we extract the feature coefficients. By matching the coefficients to the TF-IDF vocabulary, we can reveal the **Top 5 most important words** for predicting specific classes.


In [17]:
# Extract feature names from TF-IDF
feature_names = np.array(tfidf.get_feature_names_out())

# Extract coefficients from the best Logistic Regression model
coefs = best_model.coef_

# Let's look at the top 5 words for a few categories
classes_to_explain = ['Technology', 'Finance & Accounting', 'Healthcare & Wellness']

print("Explainability: Top 5 strongest predicting words per category\n")
for cls in classes_to_explain:
    class_index = list(best_model.classes_).index(cls)
    top5_indices = coefs[class_index].argsort()[-5:][::-1]
    top5_words = feature_names[top5_indices]
    print(f"Top 5 words for {cls}:")
    print(f"-> {', '.join(top5_words)}\n")


Explainability: Top 5 strongest predicting words per category

Top 5 words for Technology:
-> information technology, technology, information, technology specialist, football

Top 5 words for Finance & Accounting:
-> finance, banking, bank, ountant, financial

Top 5 words for Healthcare & Wellness:
-> healthcare, fitness, club, member, care



### Part 5: Prediction and Evaluation on Test Set
Finally, we predict on the `test_set` and evaluate the model using the **Macro-Average F1-Score** as mandated by the quality metric guidelines.
We will explicitly show the prediction for the first 5 test examples.


In [18]:
# Predict on the first 5 examples
sample_test = X_test[:5]
sample_true = y_test.iloc[:5].values
sample_pred = best_model.predict(sample_test)

print("Predictions on the first 5 Test examples:")
for i in range(5):
    print(f"Example {i+1}: True = {sample_true[i]:<30} | Predicted = {sample_pred[i]}")

# Full Test Set Evaluation
print("\n" + "="*50)
print("FULL TEST SET CLASSIFICATION REPORT")
print("="*50)

y_pred_full = best_model.predict(X_test)
print(classification_report(y_test, y_pred_full))

macro_f1 = f1_score(y_test, y_pred_full, average='macro')
print(f"\nFINAL MACRO F1-SCORE ON TEST SET: {macro_f1:.4f}")


Predictions on the first 5 Test examples:
Example 1: True = Creative & Media               | Predicted = Creative & Media
Example 2: True = Legal & HR                     | Predicted = Legal & HR
Example 3: True = Education                      | Predicted = Creative & Media
Example 4: True = Business & Operations          | Predicted = Business & Operations
Example 5: True = Business & Operations          | Predicted = Creative & Media

FULL TEST SET CLASSIFICATION REPORT
                              precision    recall  f1-score   support

       Business & Operations       0.67      0.66      0.67        74
            Creative & Media       0.67      0.76      0.72       102
                   Education       0.67      0.70      0.68        20
Engineering & Infrastructure       0.88      0.86      0.87        76
        Finance & Accounting       0.84      0.82      0.83        71
          Food & Agriculture       0.92      0.65      0.76        37
       Healthcare & Wellness   